# Bank Customer Churn – Classification & Explainable AI (XAI)

**Course:** 2400-DS2ML2 – Machine Learning 2  
**Authors:** Ronald Mjonono (473522) · Linxiao Mu (474569)

### Pipeline Overview
1. Exploratory Data Analysis (EDA)
2. Preprocessing and train–test split
3. Decision Tree classifier
4. Random Forest
5. Gradient Boosting
6. Neural Network (MLP)
7. Model evaluation & comparison
8. Baseline feature importance
9. XAI – SHAP global analysis
10. XAI – SHAP local explanations
11. XAI – SHAP dependence plots
12. XAI – SHAP cross-model comparison
13. XAI – LIME local explanations
14. XAI – Business segmentation
15. Conclusions

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import shap
import lime
import lime.lime_tabular

shap.initjs()
pd.set_option("display.max_columns", None)
sns.set_theme()

print("All libraries imported successfully.")

## 1. Data Loading and Initial Inspection

In [ ]:
df = pd.read_csv("../data/Churn_Modelling.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
id_like_cols = [c for c in ["RowNumber", "CustomerId", "Surname"] if c in df.columns]
print("Dropping:", id_like_cols)
df = df.drop(columns=id_like_cols)
df.head(3)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
target_col = "Exited"
print(df[target_col].value_counts(normalize=True).rename("proportion"))

sns.countplot(x=target_col, data=df)
plt.title("Target distribution (Exited)")
plt.show()


In [ ]:
numeric_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category"]).columns.tolist()

if target_col in numeric_features:
    numeric_features.remove(target_col)
if target_col in categorical_features:
    categorical_features.remove(target_col)

print("Numeric features   :", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
df[numeric_features].hist(bins=30, figsize=(15, 10))
plt.suptitle("Histograms of numeric features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(5, 4))
    sns.countplot(x=col, data=df)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
or col in categorical_features:
    plt.figure(figsize=(5, 4))
    churn_rate = df.groupby(col)[target_col].mean()
    sns.barplot(x=churn_rate.index, y=churn_rate.values)
    plt.title(f"Churn rate by {col}")
    plt.ylabel("Mean Exited")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_features[:6]:
    plt.figure(figsize=(5, 4))
    sns.boxplot(x=target_col, y=col, data=df)
    plt.title(f"{col} by Exited")
    plt.tight_layout()
    plt.show()

In [ ]:
corr = df[numeric_features + [target_col]].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()